# Weighted least squares on national accounts

{cite:t}`stone-champernowne-meade-1942` adjust estimates of national income, output and expenditure that violate the accounting identities between them. Each estimate has a margin of error, and the adjustment minimizes the sum of squared changes, each divided by the square of its margin. `fit` solves this problem with the `cg.LeastSquares` objective. This notebook reproduces the worked example in section IV of the paper.

In [1]:
import numpy as np
import pandas as pd

import cge_modeling as cg

In [2]:
def cell_color(value):
    if value == 0.0:
        return "color: lightgray"
    if value < 0.0:
        return "color: firebrick"
    return ""


def style_table(frame, precision, caption):
    return (frame.style
            .format(precision=precision, na_rep="")
            .map(cell_color)
            .set_caption(caption)
            .set_table_styles([{"selector": "caption",
                                "props": "caption-side: top; font-weight: bold; text-align: left"}]))

## The data

The paper has five estimates, numbered as it numbers them, with no $a_3$:

- $a_1$, national income
- $a_2$, national output
- $a_4$, national expenditure other than foreign investment
- $a_5$, foreign investment, measured from the net change in overseas assets
- $a_6$, the favorable balance of payments on current account, measured from the difference between current receipts and payments

The accounts require

$$
\begin{aligned}
a_1 &= a_2 \\
a_2 &= a_4 + a_5 \\
a_5 &= a_6
\end{aligned}
$$

The estimates satisfy the first two identities and violate the third, because the two measurements of foreign investment differ by 100.

In [3]:
names = ["a1", "a2", "a4", "a5", "a6"]
estimates = pd.Series(data=[4000.0, 4000.0, 3900.0, 100.0, 200.0], index=names)
estimates

a1    4000.0
a2    4000.0
a4    3900.0
a5     100.0
a6     200.0
dtype: float64

## Embedding the identities in a SAM

A SAM imposes one identity per account: income equals spending. Four accounts make the balance conditions exactly the paper's identities, with one estimate in each nonzero cell.

- Production pays national output $a_2$ to Output.
- Output pays national income $a_1$ to Income.
- Income pays expenditure $a_4$ to Production and foreign investment $a_5$ to Overseas.
- Overseas pays the balance of payments $a_6$ to Production.

Output balances when $a_1 = a_2$, Income when $a_1 = a_4 + a_5$, and Overseas when $a_5 = a_6$. The balance of Production, $a_4 + a_6 = a_2$, follows from the other three.

In [4]:
accounts = ["Production", "Output", "Income", "Overseas"]
cells = {"a1": ("Income", "Output"),
         "a2": ("Output", "Production"),
         "a4": ("Production", "Income"),
         "a5": ("Overseas", "Income"),
         "a6": ("Production", "Overseas")}


def read_cells(sam_frame):
    return pd.Series({name: sam_frame.loc[row, column] for name, (row, column) in cells.items()})


prior = pd.DataFrame(0.0,
                     index=accounts,
                     columns=accounts)
for name, (row, column) in cells.items():
    prior.loc[row, column] = estimates[name]

style_table(prior,
            precision=0,
            caption="Prior SAM, rows receive from columns")

,Production,Output,Income,Overseas
Production,0,0,3900,200
Output,4000,0,0,0
Income,0,4000,0,0
Overseas,0,0,100,0


## Equal margins of error

`cg.LeastSquares` minimizes

$$
\frac{1}{2} \sum_k \left( \frac{x_k - z_k}{\sigma_k} \right)^2
$$

over the nonzero cells $k$, where $z_k$ is the estimate in the prior, $x_k$ is the adjusted value, and $\sigma_k$ is the margin of error of the estimate, the paper's $\zeta$. Dividing each change by its margin makes a change of one margin cost the same in every cell, so an estimate with a small margin moves less than one with a large margin. The paper writes the weights as precisions $\pi_k = 1 / \sigma_k^2$ and calls the sum of squared standardized changes $\chi^2$, which is twice the objective of `fit`.

`sigma` is a DataFrame laid out like the prior, with each estimate's margin of error in that estimate's cell. An empty cell holds no estimate and so has no margin. `fit` reads `sigma` only at the nonzero cells of the prior, and NaN marks the rest.

With no information on the margins, the paper sets them all to one.

In [5]:
equal_sigma = pd.DataFrame(np.nan,
                           index=accounts,
                           columns=accounts)
for name, (row, column) in cells.items():
    equal_sigma.loc[row, column] = 1.0

style_table(equal_sigma,
            precision=0,
            caption="sigma: margin of error of each estimate")

,Production,Output,Income,Overseas
Production,,,1,1
Output,1,,,
Income,,1,,
Overseas,,,1,


In [6]:
sam = cg.SymbolicSAM(prior)
equal_estimate = sam.fit(objective=cg.LeastSquares(sigma=equal_sigma))
equal_adjusted = read_cells(equal_estimate.sam)
equal_chi_squared = 2.0 * equal_estimate.objective

pd.DataFrame({"estimate": estimates,
              "published": [4012.5, 4012.5, 3875.0, 137.5, 137.5],
              "reproduced": equal_adjusted}).round(3)

,estimate,published,reproduced
a1,4000.0,4012.5,4012.5
a2,4000.0,4012.5,4012.5
a4,3900.0,3875.0,3875.0
a5,100.0,137.5,137.5
a6,200.0,137.5,137.5


In [7]:
equal_chi_squared

6250.000000000075

The adjusted values and $\chi^2 = 6250$ match the paper.

## Unequal margins of error

The authors read a $\chi^2$ of 6250 on three degrees of freedom as evidence that margins of one are far too small. They then suppose a more accurate assessment of each margin: 20 for the balance of payments $a_6$, and 50 for the other four estimates. With the smaller margin, $a_6$ is the more reliable of the two measurements of foreign investment, so the adjustment moves $a_5$ more than $a_6$.

In [8]:
margins = pd.Series(data=[50.0, 50.0, 50.0, 50.0, 20.0], index=names)

sigma = pd.DataFrame(np.nan,
                     index=accounts,
                     columns=accounts)
for name, (row, column) in cells.items():
    sigma.loc[row, column] = margins[name]

style_table(sigma,
            precision=0,
            caption="sigma: margin of error of each estimate")

,Production,Output,Income,Overseas
Production,,,50,20
Output,50,,,
Income,,50,,
Overseas,,,50,


In [9]:
estimate = sam.fit(objective=cg.LeastSquares(sigma=sigma))
adjusted = read_cells(estimate.sam)
chi_squared = 2.0 * estimate.objective

pd.DataFrame({"estimate": estimates,
              "margin": margins,
              "published": [4026.4, 4026.4, 3847.5, 178.9, 178.9],
              "reproduced": adjusted}).round(3)

,estimate,margin,published,reproduced
a1,4000.0,50.0,4026.4,4026.316
a2,4000.0,50.0,4026.4,4026.316
a4,3900.0,50.0,3847.5,3847.368
a5,100.0,50.0,178.9,178.947
a6,200.0,20.0,178.9,178.947


In [10]:
published_adjusted = pd.Series(data=[4026.4, 4026.4, 3847.5, 178.9, 178.9], index=names)

pd.Series({"published": 5.288,
           "reproduced": chi_squared,
           "from the published adjusted values": (((published_adjusted - estimates) / margins) ** 2).sum()}).round(3)

published                             5.288
reproduced                            5.263
from the published adjusted values    5.263
dtype: float64

The reproduced values differ from the published ones by at most 0.13, in the last published digit of the adjusted $a_1$, $a_2$ and $a_4$. The paper's $\chi^2$ of 5.288 does not follow from its own rounded adjusted values, which give 5.263, the value at the reproduced solution.

## Margins of error of the adjusted values

The paper treats each estimate as normal around its true value, with standard deviation equal to its margin of error, and the true values satisfy the identities exactly. The adjusted values then have covariance

$$
S = V - V C^\top (C V C^\top)^{-1} C V,
$$

with $V$ the diagonal matrix of squared margins and $C$ the three identities as rows. `estimate.covariance()` returns $S$ for a `cg.LeastSquares` estimate, indexed by the row and column account of each cell. The paper reports the margins $Z = \sqrt{\operatorname{diag} S}$ and the precisions $\Pi = 1 / Z^2$.

In [11]:
covariance = estimate.covariance()
cell_margins = pd.Series(np.sqrt(np.diag(covariance)), index=covariance.index)
adjusted_margins = pd.Series({name: cell_margins[cell] for name, cell in cells.items()})

pd.DataFrame({"published margin": [29.0, 29.0, 31.0, 18.0, 18.0],
              "reproduced margin": adjusted_margins,
              "published precision": [0.00118, 0.00118, 0.00104, 0.00308, 0.00308],
              "reproduced precision": 1.0 / adjusted_margins**2}).round(5)

,published margin,reproduced margin,published precision,reproduced precision
a1,29.0,29.46898,0.00118,0.00115
a2,29.0,29.46898,0.00118,0.00115
a4,31.0,31.20391,0.00104,0.00103
a5,18.0,17.77047,0.00308,0.00317
a6,18.0,17.77047,0.00308,0.00317


## References

```{bibliography}
```

## Authors

- Created by Jesse Grabowski, November 22, 2026

## Watermark

In [12]:
%load_ext watermark
%watermark -n -u -v -iv -m -w

Last updated: Sun, 22 Nov 2026

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.18.1

Compiler    : Clang 19.1.7 
OS          : Darwin
Release     : 25.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

cge_modeling: 0.0.7.dev15+g93cb61e4e.d20261120
numpy       : 2.3.5
pandas      : 3.1.1

Watermark: 2.6.0

